# Bayesian optimization for edge shapes

Bayesian optimization using random forest surrogate.

## Example

A single campaign of Bayesian optimization.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

LABELS = [
    "Type 0",
    "Type 1a",
    "Type 1b",
    "Type 2a",
    "Type 2b",
    "Type 3a",
    "Type 3b",
]

umap = pd.read_csv("../../examples/v1/umap-embedding.csv").values
class_proba = pd.read_csv("../../examples/v1/class_proba.csv").values

bo_idxs = pd.read_csv("../../examples/v1/BO-idxs.csv")["idxs"].values
bo_pts = umap[bo_idxs]

target = class_proba.argmax(axis=1)
_, target_counts = np.unique(target, return_counts=True)

hit_types, hit_counts = np.unique(target[bo_idxs], return_counts=True)
for i in range(len(LABELS)):
    if i in hit_types:
        hit_count = hit_counts[np.where(hit_types == i)[0][0]]
    else:
        hit_count = 0
    if i < len(target_counts):
        target_count = target_counts[i]
    else:
        target_count = 0
    LABELS[i] += f" (hit: {hit_count}/{target_count})"

scatter = plt.scatter(umap[:, 0], umap[:, 1], c=target, cmap="Spectral")
cbar = plt.colorbar(scatter, boundaries=np.arange(len(np.unique(target)) + 1) - 0.5)
cbar.set_ticks(np.arange(len(np.unique(target))))
cbar.set_ticklabels(LABELS[: len(np.unique(target))])

plt.plot(*bo_pts.T, ":", color="gray", label="BO iterations")

plt.xticks([])
plt.yticks([])
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")
plt.title(f"Bayesian optimization with {len(bo_idxs)} iterations")

plt.show()

## Benchmark

Pool-based Monte-Carlo simulation and bootstrapping.

In [ ]:
def Top5p_RS(N):
    M = 0.05 * N
    P = np.empty(N, dtype=float)
    top = np.empty(N, dtype=float)

    P[0] = 0.05
    top[0] = P[0] / M
    for i in range(1, N):
        P[i] = (M - P[:i].sum()) / (N - i)
        top[i] = P[: i + 1].sum() / M
    return top

In [ ]:
import re
import pathlib
import pandas as pd
import matplotlib.ticker as ticker

ACQUISITION_FUNCTIONS = {
    "EI": r"$\alpha_\text{EI}$",
    "LCB_kappa_0.1": r"$\alpha_\text{LCB}$ ($\kappa=0.1$)",
    "LCB_kappa_1": r"$\alpha_\text{LCB}$ ($\kappa=1$)",
    "LCB_kappa_10": r"$\alpha_\text{LCB}$ ($\kappa=10$)",
    "PI": r"$\alpha_\text{PI}$",
}

pattern = re.compile(r"Bootstrap\.BO\.(?P<acq_func>.+)\.csv")

benchmark_dir = pathlib.Path("../../benchmarks/v1/")
benchmarks = list(benchmark_dir.glob("Bootstrap.BO.*.csv"))
matched = [pattern.match(path.name).group("acq_func") for path in benchmarks]

acq, dfs = [], []
for key in ACQUISITION_FUNCTIONS.keys():
    try:
        i = matched.index(key)
    except ValueError:
        continue
    acq.append(key)
    dfs.append(pd.read_csv(benchmarks[i]))
labels = [ACQUISITION_FUNCTIONS[key] for key in acq]

In [ ]:
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

fig = plt.figure(figsize=(10, 10))

ax1 = fig.add_subplot(2, 1, 1)
for df, label, color in zip(dfs, labels, colors):
    data_dict = {
        k: v.sort_values("bo_step")
        for k, v in df[df["metric"] == "top5p"].groupby("statistic")
    }
    Top5p = data_dict["mean"]
    x = np.arange(len(Top5p))

    ax1.plot(x, Top5p["value"], label=label, color=color)
    ax1.fill_between(
        x,
        data_dict["ci_low"]["value"],
        data_dict["ci_high"]["value"],
        color=color,
        alpha=0.3,
    )
ax1.plot(x, Top5p_RS(len(x)), label="Random Search", color="black", linestyle="--")

ax1.set_xscale("log")
ax1.xaxis.set_major_formatter(ticker.StrMethodFormatter("{x:.0f}"))
ax1.yaxis.set_major_formatter(ticker.StrMethodFormatter("{x:.1f}"))
ax1.set_xlabel("Iterations")
ax1.set_ylabel(r"$\operatorname{Top5\%}$")
ax1.legend()